In [ ]:

import pyodbc
import ollama


# =====================================================
# 1. SQL SERVER CONNECTION
# =====================================================

connection = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=DESKTOP-NB27H6F;"
    "DATABASE=orderssystem;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

cursor = connection.cursor()


# =====================================================
# 2. COMMON DATA FUNCTION
# =====================================================

def get_data(query):

    cursor.execute(query)

    columns = [column[0] for column in cursor.description]

    rows = cursor.fetchall()

    result = []

    for row in rows:

        result.append(dict(zip(columns, row)))

    return result


def format_data(data):

    if not data:
        return "No records found."

    return "\n".join(
        str(record) for record in data
    )


# =====================================================
# 3. OUTSTANDING EXPENSE SUMMARY
# =====================================================

def get_outstanding_summary():

    query = """

    WITH PaymentSummary AS
    (
        SELECT
            expenseid,

            SUM(
                CASE
                    WHEN status = 'Paid'
                    THEN paidamount
                    ELSE 0
                END
            ) AS PaidAmount,

            SUM(
                CASE
                    WHEN status = 'Pending'
                    THEN paidamount
                    ELSE 0
                END
            ) AS PendingAmount,

            SUM(
                CASE
                    WHEN status = 'Failed'
                    THEN paidamount
                    ELSE 0
                END
            ) AS FailedAmount

        FROM payment

        GROUP BY expenseid
    )

    SELECT

        COUNT(*) AS TotalExpenses,

        SUM(e.amount) AS TotalExpenseAmount,

        SUM(ISNULL(p.PaidAmount, 0))
            AS TotalSuccessfulPayments,

        SUM(ISNULL(p.PendingAmount, 0))
            AS TotalPendingPayments,

        SUM(ISNULL(p.FailedAmount, 0))
            AS TotalFailedPayments,

        SUM(
            e.amount - ISNULL(p.PaidAmount, 0)
        ) AS NetOutstandingAmount,

        SUM(
            CASE
                WHEN e.amount - ISNULL(p.PaidAmount, 0) > 0
                THEN e.amount - ISNULL(p.PaidAmount, 0)
                ELSE 0
            END
        ) AS TotalPositiveOutstanding,

        SUM(
            CASE
                WHEN e.amount - ISNULL(p.PaidAmount, 0) < 0
                THEN ABS(e.amount - ISNULL(p.PaidAmount, 0))
                ELSE 0
            END
        ) AS TotalOverpayment

    FROM expense e

    LEFT JOIN PaymentSummary p
        ON e.expenseid = p.expenseid;

    """

    return get_data(query)


# =====================================================
# 4. EXPENSE-WISE OUTSTANDING ANALYSIS
# =====================================================

def get_expense_outstanding():

    query = """

    WITH PaymentSummary AS
    (
        SELECT

            expenseid,

            SUM(
                CASE WHEN status = 'Paid'
                THEN paidamount ELSE 0 END
            ) AS PaidAmount,

            SUM(
                CASE WHEN status = 'Pending'
                THEN paidamount ELSE 0 END
            ) AS PendingAmount,

            SUM(
                CASE WHEN status = 'Failed'
                THEN paidamount ELSE 0 END
            ) AS FailedAmount

        FROM payment

        GROUP BY expenseid
    )

    SELECT

        e.expenseid,
        e.expensedate,
        e.description,

        ec.categoryname,

        v.vendorname,

        e.amount AS ExpenseAmount,

        ISNULL(p.PaidAmount, 0) AS PaidAmount,

        ISNULL(p.PendingAmount, 0) AS PendingAmount,

        ISNULL(p.FailedAmount, 0) AS FailedAmount,

        e.amount - ISNULL(p.PaidAmount, 0)
            AS OutstandingAmount,

        DATEDIFF(DAY, e.expensedate, GETDATE())
            AS ExpenseAgeDays,

        CASE

            WHEN e.amount - ISNULL(p.PaidAmount, 0) < 0
                THEN 'Overpaid'

            WHEN e.amount - ISNULL(p.PaidAmount, 0) = 0
                THEN 'Fully Paid'

            WHEN ISNULL(p.PaidAmount, 0) = 0
                THEN 'Unpaid'

            ELSE 'Partially Paid'

        END AS PaymentPosition

    FROM expense e

    LEFT JOIN PaymentSummary p
        ON e.expenseid = p.expenseid

    LEFT JOIN expensecategory ec
        ON e.categoryid = ec.categoryid

    LEFT JOIN vendor v
        ON e.vendorid = v.vendorid

    ORDER BY e.expensedate;

    """

    return get_data(query)


# =====================================================
# 5. VENDOR-WISE OUTSTANDING ANALYSIS
# =====================================================

def get_vendor_outstanding():

    query = """

    WITH PaymentSummary AS
    (
        SELECT

            expenseid,

            SUM(
                CASE WHEN status = 'Paid'
                THEN paidamount ELSE 0 END
            ) AS PaidAmount

        FROM payment

        GROUP BY expenseid
    )

    SELECT

        v.vendorid,
        v.vendorname,

        COUNT(e.expenseid) AS TotalExpenseRecords,

        SUM(e.amount) AS TotalExpenseAmount,

        SUM(ISNULL(p.PaidAmount, 0))
            AS TotalPaidAmount,

        SUM(
            e.amount - ISNULL(p.PaidAmount, 0)
        ) AS NetOutstandingAmount,

        SUM(
            CASE
                WHEN e.amount - ISNULL(p.PaidAmount, 0) > 0
                THEN e.amount - ISNULL(p.PaidAmount, 0)
                ELSE 0
            END
        ) AS PositiveOutstandingAmount

    FROM vendor v

    INNER JOIN expense e
        ON v.vendorid = e.vendorid

    LEFT JOIN PaymentSummary p
        ON e.expenseid = p.expenseid

    GROUP BY
        v.vendorid,
        v.vendorname

    ORDER BY PositiveOutstandingAmount DESC;

    """

    return get_data(query)


# =====================================================
# 6. CATEGORY-WISE OUTSTANDING ANALYSIS
# =====================================================

def get_category_outstanding():

    query = """

    WITH PaymentSummary AS
    (
        SELECT

            expenseid,

            SUM(
                CASE WHEN status = 'Paid'
                THEN paidamount ELSE 0 END
            ) AS PaidAmount

        FROM payment

        GROUP BY expenseid
    )

    SELECT

        ec.categoryid,
        ec.categoryname,

        COUNT(e.expenseid) AS TotalExpenseRecords,

        SUM(e.amount) AS TotalExpenseAmount,

        SUM(ISNULL(p.PaidAmount, 0))
            AS TotalPaidAmount,

        SUM(
            e.amount - ISNULL(p.PaidAmount, 0)
        ) AS NetOutstandingAmount,

        SUM(
            CASE
                WHEN e.amount - ISNULL(p.PaidAmount, 0) > 0
                THEN e.amount - ISNULL(p.PaidAmount, 0)
                ELSE 0
            END
        ) AS PositiveOutstandingAmount

    FROM expensecategory ec

    INNER JOIN expense e
        ON ec.categoryid = e.categoryid

    LEFT JOIN PaymentSummary p
        ON e.expenseid = p.expenseid

    GROUP BY
        ec.categoryid,
        ec.categoryname

    ORDER BY PositiveOutstandingAmount DESC;

    """

    return get_data(query)


# =====================================================
# 7. UNPAID EXPENSES
# =====================================================

def get_unpaid_expenses():

    query = """

    WITH PaymentSummary AS
    (
        SELECT

            expenseid,

            SUM(
                CASE WHEN status = 'Paid'
                THEN paidamount ELSE 0 END
            ) AS PaidAmount

        FROM payment

        GROUP BY expenseid
    )

    SELECT

        e.expenseid,
        e.expensedate,
        e.description,

        v.vendorname,

        ec.categoryname,

        e.amount AS ExpenseAmount,

        ISNULL(p.PaidAmount, 0) AS PaidAmount,

        e.amount - ISNULL(p.PaidAmount, 0)
            AS OutstandingAmount,

        DATEDIFF(DAY, e.expensedate, GETDATE())
            AS ExpenseAgeDays

    FROM expense e

    LEFT JOIN PaymentSummary p
        ON e.expenseid = p.expenseid

    LEFT JOIN vendor v
        ON e.vendorid = v.vendorid

    LEFT JOIN expensecategory ec
        ON e.categoryid = ec.categoryid

    WHERE
        e.amount - ISNULL(p.PaidAmount, 0) > 0

        AND ISNULL(p.PaidAmount, 0) = 0

    ORDER BY e.expensedate;

    """

    return get_data(query)


# =====================================================
# 8. PARTIALLY PAID EXPENSES
# =====================================================

def get_partially_paid_expenses():

    query = """

    WITH PaymentSummary AS
    (
        SELECT

            expenseid,

            SUM(
                CASE WHEN status = 'Paid'
                THEN paidamount ELSE 0 END
            ) AS PaidAmount

        FROM payment

        GROUP BY expenseid
    )

    SELECT

        e.expenseid,
        e.expensedate,
        e.description,

        v.vendorname,

        e.amount AS ExpenseAmount,

        p.PaidAmount,

        e.amount - p.PaidAmount
            AS OutstandingAmount,

        DATEDIFF(DAY, e.expensedate, GETDATE())
            AS ExpenseAgeDays

    FROM expense e

    INNER JOIN PaymentSummary p
        ON e.expenseid = p.expenseid

    LEFT JOIN vendor v
        ON e.vendorid = v.vendorid

    WHERE

        p.PaidAmount > 0

        AND e.amount - p.PaidAmount > 0

    ORDER BY OutstandingAmount DESC;

    """

    return get_data(query)


# =====================================================
# 9. EXPENSE AGEING ANALYSIS
# =====================================================

def get_expense_ageing():

    query = """

    WITH PaymentSummary AS
    (
        SELECT

            expenseid,

            SUM(
                CASE WHEN status = 'Paid'
                THEN paidamount ELSE 0 END
            ) AS PaidAmount

        FROM payment

        GROUP BY expenseid
    ),

    OutstandingExpenses AS
    (
        SELECT

            e.expenseid,

            e.amount - ISNULL(p.PaidAmount, 0)
                AS OutstandingAmount,

            DATEDIFF(DAY, e.expensedate, GETDATE())
                AS ExpenseAgeDays

        FROM expense e

        LEFT JOIN PaymentSummary p
            ON e.expenseid = p.expenseid

    )

    SELECT

        CASE

            WHEN ExpenseAgeDays <= 30
                THEN '0-30 Days'

            WHEN ExpenseAgeDays <= 60
                THEN '31-60 Days'

            WHEN ExpenseAgeDays <= 90
                THEN '61-90 Days'

            ELSE 'Above 90 Days'

        END AS AgeingBucket,

        COUNT(*) AS ExpenseCount,

        SUM(OutstandingAmount)
            AS NetOutstandingAmount,

        SUM(
            CASE
                WHEN OutstandingAmount > 0
                THEN OutstandingAmount
                ELSE 0
            END
        ) AS PositiveOutstandingAmount

    FROM OutstandingExpenses

    WHERE OutstandingAmount > 0

    GROUP BY

        CASE

            WHEN ExpenseAgeDays <= 30
                THEN '0-30 Days'

            WHEN ExpenseAgeDays <= 60
                THEN '31-60 Days'

            WHEN ExpenseAgeDays <= 90
                THEN '61-90 Days'

            ELSE 'Above 90 Days'

        END

    ORDER BY AgeingBucket;

    """

    return get_data(query)


# =====================================================
# 10. OVERPAID EXPENSES
# =====================================================

def get_overpaid_expenses():

    query = """

    WITH PaymentSummary AS
    (
        SELECT

            expenseid,

            SUM(
                CASE WHEN status = 'Paid'
                THEN paidamount ELSE 0 END
            ) AS PaidAmount

        FROM payment

        GROUP BY expenseid
    )

    SELECT

        e.expenseid,
        e.expensedate,
        e.description,

        v.vendorname,

        e.amount AS ExpenseAmount,

        p.PaidAmount,

        p.PaidAmount - e.amount
            AS OverpaidAmount

    FROM expense e

    INNER JOIN PaymentSummary p
        ON e.expenseid = p.expenseid

    LEFT JOIN vendor v
        ON e.vendorid = v.vendorid

    WHERE p.PaidAmount > e.amount

    ORDER BY OverpaidAmount DESC;

    """

    return get_data(query)


# =====================================================
# 11. PAYMENT STATUS ANALYSIS
# =====================================================

def get_payment_status():

    query = """

    SELECT

        status,

        COUNT(*) AS PaymentCount,

        SUM(paidamount) AS TotalPaymentAmount,

        AVG(paidamount) AS AveragePaymentAmount

    FROM payment

    GROUP BY status

    ORDER BY status;

    """

    return get_data(query)


# =====================================================
# 12. LOAD ALL ANALYSIS DATA
# =====================================================

def load_outstanding_data():

    data = {}

    data["Outstanding Summary"] = get_outstanding_summary()

    data["Expense Wise Outstanding"] = get_expense_outstanding()

    data["Vendor Wise Outstanding"] = get_vendor_outstanding()

    data["Category Wise Outstanding"] = get_category_outstanding()

    data["Unpaid Expenses"] = get_unpaid_expenses()

    data["Partially Paid Expenses"] = get_partially_paid_expenses()

    data["Expense Ageing"] = get_expense_ageing()

    data["Overpaid Expenses"] = get_overpaid_expenses()

    data["Payment Status"] = get_payment_status()

    formatted_data = ""

    for section, records in data.items():

        formatted_data += "\n\n"
        formatted_data += "========== " + section + " ==========\n"

        formatted_data += format_data(records)

    return formatted_data


# =====================================================
# 13. OUTSTANDING EXPENSE ANALYSIS AGENT
# =====================================================

def outstanding_expense_agent(user_question):

    print("\nLoading outstanding expense data...")

    expense_data = load_outstanding_data()

    print("Analysing with Llama 3.2...\n")

    response = ollama.chat(

        model="llama3.2",

        messages=[

            {
                "role": "system",

                "content": """

You are an expert Outstanding Expense and
Accounts Payable Analysis Agent.

You analyse expense and payment information
retrieved from a SQL Server accounting database.

Your responsibilities:

1. Identify unpaid expenses.
2. Identify partially paid expenses.
3. Calculate outstanding balances.
4. Analyse vendor-wise outstanding amounts.
5. Analyse category-wise outstanding amounts.
6. Identify expenses with high outstanding balances.
7. Analyse expense ageing.
8. Identify overpaid expenses.
9. Analyse Paid, Pending and Failed payments.
10. Recommend actions to improve payment management.

IMPORTANT ACCOUNTING RULES:

- Only status = 'Paid' counts as a successful payment.
- Pending payments must not reduce outstanding balances.
- Failed payments must not reduce outstanding balances.
- Outstanding = Expense Amount - Paid Amount.
- Do not add expense amounts and payment amounts together
  to calculate total expenses.
- Identify negative outstanding balances as overpayments.
- Distinguish net outstanding from positive outstanding.
- Do not call an expense overdue.
- The database does not contain payment due dates.
- Expense age is not the same as payment delay.
- Never invent figures or records.
- Use only the supplied SQL data.
- If data is unavailable, clearly mention it.
- Show financial figures to two decimal places.
- Provide practical business recommendations.

When answering, use this structure where appropriate:

1. Executive Summary
2. Outstanding Expense Position
3. Unpaid and Partially Paid Expenses
4. Vendor-wise Analysis
5. Category-wise Analysis
6. Expense Ageing
7. Overpayment Findings
8. Business Risks
9. Recommendations

Answer the user's actual question.
Do not unnecessarily repeat all analysis sections.

"""

            },

            {
                "role": "user",

                "content":

                "SQL SERVER EXPENSE DATA:\n"

                + expense_data

                + "\n\nUSER QUESTION:\n"

                + user_question

            }

        ]

    )

    return response["message"]["content"]


# =====================================================
# 14. INTERACTIVE AGENT
# =====================================================

print("\n======================================")

print("OUTSTANDING EXPENSE ANALYSIS AGENT")

print("======================================")

print("Model: Llama 3.2")

print("Database: orderssystem")

print("\nAsk your outstanding expense questions.")

print("Type 'exit' to stop the agent.\n")


while True:

    user_question = input("\nEnter your question: ")

    if user_question.lower() == "exit":

        print("\nClosing Outstanding Expense Agent...")

        break

    try:

        answer = outstanding_expense_agent(user_question)

        print("\n======================================")

        print("OUTSTANDING EXPENSE ANALYSIS")

        print("======================================\n")

        print(answer)

    except Exception as e:

        print("\nError:", e)


# =====================================================
# 15. CLOSE CONNECTION
# =====================================================

cursor.close()

connection.close()

print("\nSQL Server connection closed.")



OUTSTANDING EXPENSE ANALYSIS AGENT
Model: Llama 3.2
Database: orderssystem

Ask your outstanding expense questions.
Type 'exit' to stop the agent.




Enter your question:  Identify unpaid expenses no explanations



Loading outstanding expense data...
Analysing with Llama 3.2...

